# Group 3: Fine-tuned DistilBERT on MRPC

**Owner:** Tim McGowan

This notebook fine-tunes `distilbert-base-uncased` to decide whether two sentences are paraphrases. It trains on 100, 250, 500, 1000, 2000 and all 3,668 training pairs (3 seeds each) to build the learning curve for our research question.

**Works in two places:**
- **Colab (GPU):** for the real runs. The first cell clones our GitHub repo.
- **Locally (PyCharm / Jupyter):** for checking the code. Without a GPU it runs a short "quick test" automatically.

**Rules from the proposal:** use the published splits as-is, make every tuning decision on **validation**, and use **test** only for the final comparison.

**Outputs** (all under `results/group3_distilbert/`):
- `runs.csv`: one row per run (size, seed, accuracy, F1, time)
- `predictions/`: probabilities for every validation and test pair, keyed by `idx` (needed for stacking and the Group 3 vs 4 error table)
- `learning_curve.png` and `summary.csv`

## 1. Find the project (clone it if we're in Colab)

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/freemanchamjames/nlp_paraphrase_detection.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    PROJECT_ROOT = Path("/content/nlp_paraphrase_detection")
    if PROJECT_ROOT.exists():
        # Already cloned earlier in this session: get the latest from GitHub
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--rebase"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    # Local run: walk up from this notebook's folder until we find data/raw
    PROJECT_ROOT = Path.cwd().resolve()
    while not (PROJECT_ROOT / "data" / "raw").exists():
        if PROJECT_ROOT == PROJECT_ROOT.parent:
            raise FileNotFoundError("Couldn't find the repo. Open this notebook from inside the project folder.")
        PROJECT_ROOT = PROJECT_ROOT.parent

print("Running in Colab:", IN_COLAB)
print("Project root:    ", PROJECT_ROOT)

## 2. Imports and hardware check

If this cell says `No module named ...` locally, run `pip install -r requirements.txt` in the PyCharm terminal. Colab already has everything.

In Colab, turn on the GPU first: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
import json
import time
import shutil
import tempfile
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)
import transformers

warnings.filterwarnings("ignore", category=FutureWarning)

HAS_GPU = torch.cuda.is_available()
print("torch", torch.__version__, "| transformers", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if HAS_GPU else "none (CPU only)")

## 3. Settings

Everything you might change lives here.

- `QUICK_TEST` is on automatically when there's no GPU. It trains one tiny run (100 pairs, 1 epoch) to prove the code works, and saves to a separate `_quick_test` folder so it never mixes with real results.
- Smaller training sets get more epochs, since 100 pairs is only about 7 batches per epoch. Early stopping ends a run once validation F1 stops improving.

In [ ]:
MODEL_NAME    = "distilbert-base-uncased"
MAX_LENGTH    = 128      # MRPC pairs are short; 128 tokens covers almost all of them
LEARNING_RATE = 2e-5
BATCH_SIZE    = 16
WEIGHT_DECAY  = 0.01
WARMUP_RATIO  = 0.1
PATIENCE      = 3        # epochs with no validation-F1 gain before stopping

TRAIN_SIZES = [100, 250, 500, 1000, 2000, "full"]
SEEDS       = [42, 43, 44]

def max_epochs_for(n_pairs):
    if n_pairs <= 250:
        return 20
    if n_pairs <= 1000:
        return 10
    return 5

QUICK_TEST = not HAS_GPU   # set to False to force full runs (slow on CPU)

if QUICK_TEST:
    TRAIN_SIZES = [100]
    SEEDS = [42]
    max_epochs_for = lambda n_pairs: 1
    OUT_DIR = PROJECT_ROOT / "results" / "group3_distilbert" / "_quick_test"
else:
    OUT_DIR = PROJECT_ROOT / "results" / "group3_distilbert"

PRED_DIR = OUT_DIR / "predictions"
PRED_DIR.mkdir(parents=True, exist_ok=True)
RUNS_CSV = OUT_DIR / "runs.csv"

# Checkpoints are big (~250 MB each), so they go to a temp folder outside the repo
CKPT_ROOT = Path(tempfile.gettempdir()) / "group3_checkpoints"

print("QUICK_TEST:", QUICK_TEST)
print("Saving results to:", OUT_DIR)

## 4. Load the shared data from the repo

These are the CSVs made by `data/prepare_mrpc.py` (the published GLUE splits, untouched).

We check row counts and label counts instead of the SHA-256 values in `metadata.json`. Those hashes were made on Windows, where lines end in `\r\n`. Git stores the files with plain `\n` endings, so on Colab or a Mac the hashes won't match even though the data is identical.

In [ ]:
RAW_DIR = PROJECT_ROOT / "data" / "raw"
train_df = pd.read_csv(RAW_DIR / "train.csv")
val_df   = pd.read_csv(RAW_DIR / "validation.csv")
test_df  = pd.read_csv(RAW_DIR / "test.csv")

# Expected (rows, label-1 count, label-0 count) for the published MRPC splits
EXPECTED = {"train": (3668, 2474, 1194), "validation": (408, 279, 129), "test": (1725, 1147, 578)}

for name, df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    got = (len(df), int((df.label == 1).sum()), int((df.label == 0).sum()))
    status = "OK" if got == EXPECTED[name] else f"MISMATCH, expected {EXPECTED[name]}"
    print(f"{name:10s} rows={got[0]:5d}  paraphrase={got[1]:5d}  not={got[2]:5d}  "
          f"({got[1] / got[0]:.1%} paraphrase)  {status}")
    assert got == EXPECTED[name], f"{name} doesn't match the published split"

## 5. How DistilBERT sees a sentence pair

Both sentences go in together as `[CLS] sentence1 [SEP] sentence2 [SEP]`. Because they share one input, attention can compare words across the two sentences in every layer. That's Group 3's answer to the proposal's question, "where do the two sentences meet?"

MRPC text has extra spaces around punctuation (`Dominick 's`, `$ 2.5`). We **don't** clean it for Group 3, because DistilBERT's own tokenizer splits punctuation anyway (the proposal says the transformer uses its own tokenizer).

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

example = train_df.iloc[1]
enc = tokenizer(example.sentence1, example.sentence2)
print("Sentence 1:", example.sentence1)
print("Sentence 2:", example.sentence2)
print("Label:", example.label)
print("\nWhat the model sees:")
print(tokenizer.convert_ids_to_tokens(enc["input_ids"]))

# How many pairs would be cut off at MAX_LENGTH?
lengths = [len(tokenizer(a, b)["input_ids"]) for a, b in zip(train_df.sentence1, train_df.sentence2)]
print(f"\nLongest pair: {max(lengths)} tokens | pairs over {MAX_LENGTH}: {sum(l > MAX_LENGTH for l in lengths)}")

## 6. Helper functions

In [ ]:
def to_hf_dataset(df):
    """Turn a DataFrame into tokenized input the Trainer can use."""
    ds = Dataset.from_pandas(df[["sentence1", "sentence2", "label"]].reset_index(drop=True))
    return ds.map(
        lambda b: tokenizer(b["sentence1"], b["sentence2"], truncation=True, max_length=MAX_LENGTH),
        batched=True,
        remove_columns=["sentence1", "sentence2"],
    )

val_ds  = to_hf_dataset(val_df)
test_ds = to_hf_dataset(test_df)


def sample_train(n, seed):
    """Stratified sample: keeps the same ~68/32 label mix as the full training set."""
    if n == "full" or n >= len(train_df):
        return train_df
    subset, _ = train_test_split(train_df, train_size=n, stratify=train_df["label"], random_state=seed)
    return subset


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {"accuracy": accuracy_score(labels, preds), "f1": f1_score(labels, preds)}


def predict_and_save(trainer, ds, df, split, n, seed):
    """Get P(paraphrase) for every pair and save it, keyed by idx."""
    logits = trainer.predict(ds).predictions
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    preds = (probs >= 0.5).astype(int)
    out = pd.DataFrame({"idx": df["idx"].values, "label": df["label"].values,
                        "prob_paraphrase": probs.round(6), "pred": preds})
    out.to_csv(PRED_DIR / f"{split}_n{n}_seed{seed}.csv", index=False)
    return accuracy_score(out.label, out.pred), f1_score(out.label, out.pred)

## 7. One training run

`run_one(n, seed)` trains a fresh DistilBERT on `n` pairs, keeps the epoch with the best **validation F1**, then saves predictions for validation and test.

Test scores are recorded for the final comparison only. Don't use them to pick settings.

In [ ]:
def run_one(n, seed):
    sub = sample_train(n, seed)
    n_pairs = len(sub)
    max_epochs = max_epochs_for(n_pairs)
    set_seed(seed)

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
    ckpt_dir = CKPT_ROOT / f"n{n_pairs}_seed{seed}"

    # Warmup = first 10% of all training steps (works on old and new transformers versions)
    total_steps = -(-n_pairs // BATCH_SIZE) * max_epochs
    warmup_steps = int(WARMUP_RATIO * total_steps)

    args = TrainingArguments(
        output_dir=str(ckpt_dir),
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=64,
        num_train_epochs=max_epochs,
        weight_decay=WEIGHT_DECAY,
        warmup_steps=warmup_steps,
        fp16=HAS_GPU,
        seed=seed,
        logging_strategy="epoch",
        report_to="none",
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=to_hf_dataset(sub),
        eval_dataset=val_ds,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
    )

    start = time.time()
    trainer.train()
    seconds = time.time() - start

    epochs_run = trainer.state.epoch
    best_epoch = None
    for entry in trainer.state.log_history:
        if "eval_f1" in entry and entry["eval_f1"] == trainer.state.best_metric:
            best_epoch = entry["epoch"]
            break

    val_acc, val_f1 = predict_and_save(trainer, val_ds, val_df, "validation", n_pairs, seed)
    test_acc, test_f1 = predict_and_save(trainer, test_ds, test_df, "test", n_pairs, seed)

    shutil.rmtree(ckpt_dir, ignore_errors=True)   # free disk space
    del trainer, model
    if HAS_GPU:
        torch.cuda.empty_cache()

    return {
        "n_train": n_pairs, "seed": seed,
        "epochs_run": round(float(epochs_run), 2),
        "best_epoch": best_epoch, "max_epochs": max_epochs,
        "val_accuracy": round(val_acc, 4), "val_f1": round(val_f1, 4),
        "test_accuracy": round(test_acc, 4), "test_f1": round(test_f1, 4),
        "train_seconds": round(seconds, 1),
        "model": MODEL_NAME, "lr": LEARNING_RATE, "batch_size": BATCH_SIZE,
        "finished_at": datetime.now().isoformat(timespec="seconds"),
    }

## 8. Run the learning curve

This cell **picks up where it left off**. Any (size, seed) already in `runs.csv` is skipped, so if Colab disconnects you can rerun it after pulling the latest results from GitHub.

Rough time on a Colab T4: about 30–45 minutes for all 18 runs.

In [ ]:
runs = pd.read_csv(RUNS_CSV) if RUNS_CSV.exists() else pd.DataFrame()
done = set(zip(runs.get("n_train", []), runs.get("seed", [])))

for n in TRAIN_SIZES:
    n_pairs = len(train_df) if n == "full" else n
    for seed in SEEDS:
        if (n_pairs, seed) in done:
            print(f"skip  n={n_pairs:5d} seed={seed} (already in runs.csv)")
            continue
        print(f"\n=== training n={n_pairs} seed={seed} ===")
        result = run_one(n, seed)
        runs = pd.concat([runs, pd.DataFrame([result])], ignore_index=True)
        runs.to_csv(RUNS_CSV, index=False)   # save after every run
        print(f"done  val F1={result['val_f1']:.3f}  test F1={result['test_f1']:.3f}  "
              f"({result['train_seconds']:.0f}s)")

runs

## 9. Summary table and learning curve

The mean and standard deviation across seeds for each training size. The shaded band shows how much results move between seeds. When two points' bands overlap, the difference may just be noise.

Once Group 4 has Qwen's test F1 scores, enter them below and they'll show up as horizontal lines. Where the DistilBERT curve crosses them is the answer to our research question.

In [ ]:
QWEN_ZERO_SHOT_TEST_F1 = None   # e.g. 0.78, from Group 4
QWEN_FEW_SHOT_TEST_F1  = None

runs = pd.read_csv(RUNS_CSV)
summary = (runs.groupby("n_train")
               .agg(runs=("seed", "count"),
                    val_f1_mean=("val_f1", "mean"), val_f1_std=("val_f1", "std"),
                    test_f1_mean=("test_f1", "mean"), test_f1_std=("test_f1", "std"),
                    test_acc_mean=("test_accuracy", "mean"))
               .round(4).reset_index())
summary.to_csv(OUT_DIR / "summary.csv", index=False)
display(summary)

# Baseline to beat: always guessing "paraphrase"
always_yes_f1 = f1_score(test_df.label, np.ones(len(test_df), dtype=int))

fig, ax = plt.subplots(figsize=(7, 4.2))
x = summary.n_train
mean, std = summary.test_f1_mean, summary.test_f1_std.fillna(0)
ax.plot(x, mean, marker="o", color="#2a6fdb", label="DistilBERT (mean of seeds)")
ax.fill_between(x, mean - std, mean + std, color="#2a6fdb", alpha=0.18, label="± 1 std across seeds")
ax.axhline(always_yes_f1, color="#888888", ls=":", lw=1.2, label=f'Always "paraphrase" ({always_yes_f1:.3f})')
if QWEN_ZERO_SHOT_TEST_F1 is not None:
    ax.axhline(QWEN_ZERO_SHOT_TEST_F1, color="#d1495b", ls="--", lw=1.4, label="Qwen zero-shot")
if QWEN_FEW_SHOT_TEST_F1 is not None:
    ax.axhline(QWEN_FEW_SHOT_TEST_F1, color="#e8873a", ls="--", lw=1.4, label="Qwen few-shot")
ax.set_xscale("log")
ax.set_xticks(list(x))
ax.set_xticklabels([str(v) for v in x])
ax.minorticks_off()
ax.set_xlabel("Labeled training pairs (log scale)")
ax.set_ylabel("Test F1")
ax.set_title("DistilBERT learning curve on MRPC")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3)
ax.legend(frameon=False, fontsize=8, loc="lower right")
fig.tight_layout()
fig.savefig(OUT_DIR / "learning_curve.png", dpi=150)
plt.show()

## 10. Save results to GitHub

Only the small result files get committed (`runs.csv`, `summary.csv`, `predictions/`, `learning_curve.png`). Model weights stay out of the repo.

**Locally (PyCharm):** Git → Commit, tick `results/group3_distilbert`, write a message, then Commit and Push. Or in the terminal:
```
git pull --rebase
git add results/group3_distilbert
git commit -m "Group 3: DistilBERT learning curve results"
git push
```

**In Colab:** run the cell below. One-time setup:
1. On GitHub: Settings → Developer settings → Personal access tokens → **Fine-grained token**, with access to this repo and **Contents: Read and write**.
2. In Colab: click the 🔑 **Secrets** icon in the left sidebar, add `GITHUB_TOKEN` with the token as its value, and turn on notebook access.

Never paste the token into the notebook itself.

In [ ]:
GIT_NAME  = "Tim McGowan"
GIT_EMAIL = "tbm0929@gmail.com"   # the email on your GitHub account

if IN_COLAB and not QUICK_TEST:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    push_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")

    def git(*cmd):
        r = subprocess.run(["git", "-C", str(PROJECT_ROOT), *cmd], capture_output=True, text=True)
        print((r.stdout + r.stderr).replace(token, "***").strip())
        return r.returncode

    git("config", "user.name", GIT_NAME)
    git("config", "user.email", GIT_EMAIL)
    git("add", "results/group3_distilbert")
    git("commit", "-m", f"Group 3: DistilBERT results ({len(runs)} runs)")
    git("pull", "--rebase", push_url, "main")
    if git("push", push_url, "HEAD:main") == 0:
        print("Pushed to GitHub.")
else:
    print("Local run or quick test: commit from PyCharm instead (see the notes above).")